# Lab 3 — Prompt คือ code: เก็บเป็นเวอร์ชัน แล้วเล่นงานเดิมซ้ำ (EP09)

สองประโยคจากตำรา §18.10 ที่แล็บนี้ทำให้เห็นจริง:

- **Prompts are code** — เก็บเวอร์ชันไว้ ทดสอบ และประกอบขึ้นจากส่วนย่อย
- **Regression testing** (วิธีที่ ⑤) — เล่นงานจริงที่เคยรันไว้ซ้ำ แล้วตรวจว่าผลไม่แย่ลงหลังแก้

| ขั้น | ในแล็บนี้ |
|---|---|
| ประกอบ prompt | system prompt สร้างจากไฟล์ส่วนย่อย `role` · `rules` · `cite` · `format` และมีเลขเวอร์ชัน + hash |
| บันทึก | รันงาน 5 ข้อด้วย v1 กับ model จริง เก็บ trace ลงไฟล์ JSONL |
| เล่นซ้ำ | รันงานเดิมด้วย v2 แล้วเทียบกับ trace เดิมทีละข้อ: ดีขึ้น · แย่ลง · เท่าเดิม |
| จับ regression | v3 ที่ "ประหยัด" กว่า — ดูว่ามันทำอะไรพัง |

> อ้างอิง: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §18.10 — CC BY-SA 4.0
> tool ในแล็บนี้ใช้คลัง paper จำลอง แต่คำตอบของ model ทุกคำตอบมาจากการรันจริง

## 0. เตรียมเครื่องก่อนเริ่ม

| สิ่งที่ต้องมี | รายละเอียด |
|---|---|
| Python 3.10 ขึ้นไป | หรือใช้ Google Colab ซึ่งมีให้แล้ว |
| OpenRouter API key | key เดิมจาก Lab 1 (แล็บนี้ใช้ไม่ถึง 1 เซ็นต์ดอลลาร์) |
| Packages | `openai`, `python-dotenv`, `tokenizers` (ติดตั้งใน cell ถัดไป) |

**ใส่ API key**
- **Colab:** ไอคอน 🔑 Secrets → ชื่อ `OPENROUTER_API_KEY` → เปิด *Notebook access*
- **เครื่องตัวเอง:** สร้างไฟล์ `.env` ในโฟลเดอร์เดียวกับ notebook แล้วใส่บรรทัด `OPENROUTER_API_KEY=sk-or-...`

⚠️ อย่าใส่ key ลงในโค้ดโดยตรง และอย่าแชร์ไฟล์ `.env`

In [ ]:
%pip install -q "openai>=1.50" python-dotenv "tokenizers==0.23.3"

## 1. harness จาก Lab 1

สาม cell ถัดไปคือ harness ตัวเดียวกับ Lab 2 (รับ `client` เป็น argument ตามตำรา) — วางไว้ให้แล้ว ไม่ต้องพิมพ์ใหม่

In [ ]:
import asyncio, hashlib, json, logging, os, time
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable, Optional
from openai import AsyncOpenAI
from tokenizers import Tokenizer

MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}
tok = Tokenizer.from_pretrained(MODEL)
logging.basicConfig(level=logging.WARNING, format="%(message)s", force=True)
logger = logging.getLogger("agent_harness")
logger.setLevel(logging.INFO)

class Role(str, Enum):
    SYSTEM = "system"
    USER = "user"
    ASSISTANT = "assistant"
    TOOL = "tool"

@dataclass
class Message:
    role: Role
    content: str
    tool_calls: Optional[list[dict]] = None
    tool_call_id: Optional[str] = None

    def to_api_dict(self) -> dict:
        d = {"role": self.role.value, "content": self.content or None}
        if self.tool_calls:
            d["tool_calls"] = self.tool_calls
        if self.tool_call_id:
            d["tool_call_id"] = self.tool_call_id
        return d

@dataclass
class ToolDefinition:
    name: str
    description: str
    parameters: dict
    handler: Callable
    requires_approval: bool = False

    def to_api_dict(self) -> dict:
        return {"type": "function",
                "function": {"name": self.name, "description": self.description,
                             "parameters": self.parameters}}

def count_tokens(text: str) -> int:
    return len(tok.encode(text).ids)

class ContextManager:
    BUDGET_FRACTIONS = {"system": 0.10, "memory": 0.20, "tools": 0.10,
                        "history": 0.50, "reserved": 0.10}

    def __init__(self, max_tokens: int):
        self.max_tokens = max_tokens
        self.history: list[Message] = []
        self.system_msg: Optional[Message] = None

    def count_message_tokens(self, msg: Message) -> int:
        return count_tokens(msg.content or "") + 4

    def total_history_tokens(self) -> int:
        return sum(self.count_message_tokens(m) for m in self.history)

    def history_budget(self) -> int:
        return int(self.max_tokens * self.BUDGET_FRACTIONS["history"])

    def add_message(self, msg: Message) -> None:
        self.history.append(msg)
        self._enforce_budget()

    def _enforce_budget(self) -> None:
        while self.total_history_tokens() > self.history_budget() and len(self.history) > 2:
            dropped = self.history.pop(1)
            if dropped.tool_calls:
                while len(self.history) > 1 and self.history[1].role == Role.TOOL:
                    self.history.pop(1)

    def preflight_check(self, tool_tokens: int) -> bool:
        sys_tokens = self.count_message_tokens(self.system_msg) if self.system_msg else 0
        total = sys_tokens + tool_tokens + self.total_history_tokens()
        limit = self.max_tokens - int(self.max_tokens * self.BUDGET_FRACTIONS["reserved"])
        if total > limit:
            logger.warning("Context overflow: %d > %d", total, limit)
        return total <= limit

    def build_messages(self) -> list[dict]:
        msgs = [self.system_msg.to_api_dict()] if self.system_msg else []
        return msgs + [m.to_api_dict() for m in self.history]

In [ ]:
class ToolExecutor:
    MAX_OUTPUT_TOKENS = 2000
    MAX_RETRIES = 3
    TIMEOUT = 30.0

    def __init__(self, tools: list[ToolDefinition], approval_callback: Optional[Callable] = None):
        self.tools = {t.name: t for t in tools}
        self.approval = approval_callback

    async def execute(self, tool_name: str, args: dict) -> str:
        tool = self.tools.get(tool_name)
        if not tool:
            return f"Error: unknown tool '{tool_name}'"
        if tool.requires_approval and self.approval:
            if not await self.approval(tool_name, args):
                return "Action rejected by human reviewer."
        for attempt in range(self.MAX_RETRIES):
            try:
                result = await asyncio.wait_for(self._call(tool, args), timeout=self.TIMEOUT)
                return self._truncate(result)
            except asyncio.TimeoutError:
                logger.warning("Tool %s timed out (attempt %d)", tool_name, attempt + 1)
                if attempt == self.MAX_RETRIES - 1:
                    return f"Error: tool '{tool_name}' timed out"
                await asyncio.sleep(2 ** attempt)
            except Exception as exc:
                logger.error("Tool %s error: %s", tool_name, exc)
                if attempt == self.MAX_RETRIES - 1:
                    return f"Error: {exc}"
                await asyncio.sleep(2 ** attempt)
        return "Error: max retries exceeded"

    async def _call(self, tool: ToolDefinition, args: dict) -> str:
        if asyncio.iscoroutinefunction(tool.handler):
            return str(await tool.handler(**args))
        loop = asyncio.get_running_loop()
        return str(await loop.run_in_executor(None, lambda: tool.handler(**args)))

    def _truncate(self, text: str) -> str:
        enc = tok.encode(text)
        if len(enc.ids) <= self.MAX_OUTPUT_TOKENS:
            return text
        end = enc.offsets[self.MAX_OUTPUT_TOKENS - 1][1]
        return text[:end] + "\n[... output truncated ...]"

class LoopDetector:
    def __init__(self, window: int = 5, max_repeats: int = 2):
        self.window = window
        self.max_repeats = max_repeats
        self.action_hashes: list[str] = []

    def record(self, tool_name: str, args: dict) -> bool:
        h = hashlib.md5(f"{tool_name}:{json.dumps(args, sort_keys=True)}".encode()).hexdigest()
        self.action_hashes.append(h)
        recent = self.action_hashes[-self.window:]
        if recent.count(h) >= self.max_repeats:
            logger.warning("Loop detected: %s called %d times", tool_name, recent.count(h))
            return True
        return False

In [ ]:
class AgentHarness:
    MAX_ITERATIONS = 50

    def __init__(self, system_prompt: str, tools: list[ToolDefinition],
                 max_tokens: int = 256_000, approval_cb: Optional[Callable] = None, client=None):
        self.client = client or AsyncOpenAI(base_url="https://openrouter.ai/api/v1",
                                            api_key=os.environ.get("OPENROUTER_API_KEY", "no-key"))
        self.ctx_mgr = ContextManager(max_tokens)
        self.executor = ToolExecutor(tools, approval_cb)
        self.loop_det = LoopDetector()
        self.tools = tools
        self.ctx_mgr.system_msg = Message(Role.SYSTEM, system_prompt)
        self.usage = {"in": 0, "out": 0}

    async def run(self, user_input: str) -> str:
        run_id = hashlib.md5(f"{time.time()}:{user_input}".encode()).hexdigest()[:8]
        start_ts = time.monotonic()
        logger.info("[%s] Starting run: %s", run_id, user_input[:80])
        self.ctx_mgr.add_message(Message(Role.USER, user_input))
        tool_defs = [t.to_api_dict() for t in self.tools]
        tool_tokens = sum(count_tokens(json.dumps(t)) for t in tool_defs)
        msg = None

        for iteration in range(self.MAX_ITERATIONS):
            if not self.ctx_mgr.preflight_check(tool_tokens):
                logger.error("[%s] Context overflow at iter %d", run_id, iteration)
                return "I've run out of context space. Please start a new conversation."
            try:
                response = await self.client.chat.completions.create(
                    model=MODEL, messages=self.ctx_mgr.build_messages(),
                    tools=tool_defs or None, tool_choice="auto", temperature=0.0, extra_body=EXTRA)
            except Exception as exc:
                logger.error("[%s] LLM call failed: %s", run_id, exc)
                return f"I encountered an error: {exc}"
            self.usage["in"] += response.usage.prompt_tokens
            self.usage["out"] += response.usage.completion_tokens
            msg, finish = response.choices[0].message, response.choices[0].finish_reason
            self.ctx_mgr.add_message(Message(Role.ASSISTANT, msg.content or "",
                tool_calls=[tc.model_dump() for tc in msg.tool_calls] if msg.tool_calls else None))
            if finish == "stop" or not msg.tool_calls:
                logger.info("[%s] Done in %d iters, %.2fs", run_id, iteration + 1, time.monotonic() - start_ts)
                return msg.content or "Task complete."

            results = await self._execute_tool_calls(msg.tool_calls, run_id)
            for tc in msg.tool_calls:
                if self.loop_det.record(tc.function.name, json.loads(tc.function.arguments)):
                    return "I seem to be stuck in a loop. Please clarify your request."
            for tool_call_id, result in results.items():
                self.ctx_mgr.add_message(Message(Role.TOOL, result, tool_call_id=tool_call_id))

        logger.warning("[%s] Max iterations reached", run_id)
        return ("I reached the maximum number of steps without completing the task. "
                "Here is what I found so far: " + (msg.content or ""))

    async def _execute_tool_calls(self, tool_calls: list, run_id: str) -> dict[str, str]:
        tasks = {}
        for tc in tool_calls:
            try:
                args = json.loads(tc.function.arguments)
            except json.JSONDecodeError:
                args = {}
            logger.info("[%s] Tool call: %s(%s)", run_id, tc.function.name, args)
            tasks[tc.id] = self.executor.execute(tc.function.name, args)
        t0 = time.monotonic()
        results = await asyncio.gather(*tasks.values(), return_exceptions=True)
        logger.info("[%s] %d tool(s) done in %.1fs", run_id, len(tasks), time.monotonic() - t0)
        return {tid: (f"Error: {r}" if isinstance(r, Exception) else r)
                for tid, r in zip(tasks.keys(), results)}

In [ ]:
def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = AsyncOpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(),
                     max_retries=5, timeout=60)
logger.setLevel(logging.WARNING)

## 2. tool สองตัว

คลัง paper จำลองหกเรื่องเดิม · `search_papers` และ `read_paper` จาก Lab 1 (แล็บนี้ไม่บันทึกรายงาน จึงไม่มี `save_report`)

In [ ]:
LIBRARY = {
    "P1": ("Informer (2021)", "ProbSparse self-attention makes long-sequence time-series forecasting faster."),
    "P2": ("PatchTST (2023)", "Splits each series into patches of 16 steps; each channel is handled on its own."),
    "P3": ("TimesFM (2024)", "A decoder-only foundation model that forecasts new series zero-shot."),
    "P4": ("Autoformer (2021)", "Replaces attention with auto-correlation and decomposes trend and season."),
    "P5": ("DLinear (2023)", "A single linear layer that beats many transformers on long-term forecasting."),
    "P6": ("BERT (2019)", "Pre-trains a bidirectional transformer on text with masked language modelling."),
}

async def search_papers(query: str, max_results: int = 3) -> str:
    await asyncio.sleep(0.5)
    words = [w.rstrip("s") for w in query.lower().replace("-", " ").split() if len(w) > 3]
    hits = [(pid, t) for pid, (t, a) in LIBRARY.items() if any(w in (t + " " + a).lower() for w in words)]
    return json.dumps([{"id": pid, "title": t} for pid, t in hits[:max_results]])

async def read_paper(paper_id: str) -> str:
    await asyncio.sleep(1.5)
    title, abstract = LIBRARY[paper_id]
    return json.dumps({"id": paper_id, "title": title, "abstract": abstract})

TOOLS = [
    ToolDefinition("search_papers", "Search the paper library. Returns ids and titles.",
        {"type": "object", "properties": {"query": {"type": "string"},
         "max_results": {"type": "integer", "default": 3}}, "required": ["query"]}, search_papers),
    ToolDefinition("read_paper", "Read one paper's abstract by its id (P1, P2, ...).",
        {"type": "object", "properties": {"paper_id": {"type": "string"}}, "required": ["paper_id"]}, read_paper),
]

## 3. Prompt คือ code — ประกอบจากส่วนย่อย มีเวอร์ชัน

ตำราสรุปว่า prompt ต้อง *version-control, test, assemble modularly from components*
- แต่ละส่วนเป็นไฟล์ของมันเอง (ในโปรเจกต์จริงอยู่ใน git — ที่นี่เราเขียนไฟล์จาก notebook เพื่อให้แล็บรันได้ด้วยตัวเอง)
- `versions.json` บอกว่าแต่ละเวอร์ชันใช้ส่วนไหนบ้าง
- ทุก prompt ได้ **id = เวอร์ชัน + hash ของข้อความ** — ถ้ามีคนแก้ไฟล์แม้ตัวอักษรเดียว hash ก็เปลี่ยน

In [ ]:
import difflib, re
from pathlib import Path

PROMPTS = Path("prompts")
PROMPTS.mkdir(exist_ok=True)
PARTS = {
    "role":   "You are a research assistant for a university course.",
    "rules":  "Use the tools to find and read papers before answering. "
              "When you need several papers, request them all in the same turn.",
    "cite":   "Always cite paper ids like [P2].",
    "format": "Answer in 2-4 sentences.",
}
for name, text in PARTS.items():
    (PROMPTS / f"{name}.txt").write_text(text + "\n")

VERSIONS = {
    "v1": ["role", "rules", "format"],
    "v2": ["role", "rules", "cite", "format"],
}
(PROMPTS / "versions.json").write_text(json.dumps(VERSIONS, indent=2))

def build_prompt(version: str) -> tuple[str, str]:
    parts = json.loads((PROMPTS / "versions.json").read_text())[version]
    text = " ".join((PROMPTS / f"{p}.txt").read_text().strip() for p in parts)
    prompt_id = f"{version}-{hashlib.sha256(text.encode()).hexdigest()[:8]}"
    return text, prompt_id

In [ ]:
v1, id1 = build_prompt("v1")
v2, id2 = build_prompt("v2")
print(id1, "→", id2)
sentences = lambda t: re.split(r"(?<=[.])\s+", t)
for line in difflib.unified_diff(sentences(v1), sentences(v2), "v1", "v2", lineterm="", n=0):
    print(line)

## 4. บันทึก trace ของ v1

**trace** = บันทึกหนึ่งงาน: งานอะไร · prompt id · tool ที่เรียก · paper ที่อ่าน · paper ที่อ้างอิง · คำตอบ · token · เวลา
เก็บเป็น JSONL (หนึ่งบรรทัดต่อหนึ่งงาน) — ใน production นี่คือ log ของงานจริงที่ผู้ใช้ส่งมา

In [ ]:
TASKS = [
    {"task": "Which paper splits each series into patches?", "expect": ["P2"]},
    {"task": "Compare Informer and Autoformer.", "expect": ["P1", "P4"]},
    {"task": "What does TimesFM do?", "expect": ["P3"]},
    {"task": "Is a single linear layer enough for long-term forecasting?", "expect": ["P5"]},
    {"task": "Which paper uses masked language modelling?", "expect": ["P6"]},
]

async def run_task(version: str, item: dict) -> dict:
    prompt, prompt_id = build_prompt(version)
    h = AgentHarness(prompt, TOOLS, client=client)
    t0 = time.monotonic()
    answer = await h.run(item["task"])
    calls = [(tc["function"]["name"], json.loads(tc["function"]["arguments"]))
             for m in h.ctx_mgr.history if m.tool_calls for tc in m.tool_calls]
    return {"task": item["task"], "prompt": prompt_id,
            "tools": [name for name, _ in calls],
            "read": sorted(a["paper_id"] for name, a in calls if name == "read_paper"),
            "cites": sorted(set(re.findall(r"\[(P\d)\]", answer))),
            "answer": answer, "tokens": h.usage,
            "seconds": round(time.monotonic() - t0, 1)}

TRACES = Path("traces")
TRACES.mkdir(exist_ok=True)

async def record(version: str) -> list[dict]:
    traces = [await run_task(version, item) for item in TASKS]
    with open(TRACES / f"{version}.jsonl", "w") as f:
        for t in traces:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    return traces

def load(version: str) -> list[dict]:
    return [json.loads(line) for line in open(TRACES / f"{version}.jsonl")]

In [ ]:
def show(traces):
    for t in traces:
        print(f"{t['prompt']}  {t['seconds']:>4}s  tools={t['tools']}  read={t['read']}  cites={t['cites']}")

base = await record("v1")
show(base)
print()
print(base[1]["answer"])

## 5. เล่นซ้ำด้วย v2 แล้วเทียบ

ตำรา (วิธีที่ ⑤): *replay past production traces and verify that outputs do not regress after changes*

แต่ละงานมีสองข้อตรวจที่ไม่ต้องใช้คนอ่าน:
- `read` — อ่าน paper ที่ควรอ่านครบไหม
- `cited` — อ้างอิง paper ที่ควรอ้างครบไหม

ข้อตรวจไหน **ผ่านในของเดิมแต่ไม่ผ่านในของใหม่** = regression

In [ ]:
def checks(trace: dict, expect: list[str]) -> dict[str, bool]:
    return {"read": set(expect) <= set(trace["read"]),
            "cited": set(expect) <= set(trace["cites"])}

def compare(old: list[dict], new: list[dict]) -> list[tuple]:
    rows = []
    for item, a, b in zip(TASKS, old, new):
        before, after = checks(a, item["expect"]), checks(b, item["expect"])
        lost = [k for k in before if before[k] and not after[k]]
        gained = [k for k in before if after[k] and not before[k]]
        verdict = "REGRESSED" if lost else "improved" if gained else "same"
        rows.append((verdict, item["task"], lost or gained))
        print(f"{verdict:9}  {item['task'][:48]:48}  {', '.join(lost or gained)}")
    return rows

def cost(traces: list[dict]) -> str:
    t_in = sum(t["tokens"]["in"] for t in traces)
    t_out = sum(t["tokens"]["out"] for t in traces)
    return f"{t_in:,} in + {t_out:,} out ≈ ${(t_in * 0.09 + t_out * 0.28) / 1e6:.5f}"

In [ ]:
new = await record("v2")
show(new)
print()
rows = compare(load("v1"), new)
print()
print("v1:", cost(load("v1")))
print("v2:", cost(new))

## 6. v3 — prompt ที่ "ประหยัด" กว่า

สมมติเพื่อนร่วมทีมอยากลดค่า token ขาออก จึงเปลี่ยนส่วน format เป็น "ตอบประโยคเดียว" — ดูดีบนกระดาษ ลองเล่นซ้ำดูก่อนปล่อย

In [ ]:
(PROMPTS / "format_short.txt").write_text("Answer in one short sentence.\n")
VERSIONS["v3"] = ["role", "rules", "cite", "format_short"]
(PROMPTS / "versions.json").write_text(json.dumps(VERSIONS, indent=2))

v3 = await record("v3")
show(v3)
print()
rows = compare(load("v2"), v3)
print()
print("v2:", cost(load("v2")))
print("v3:", cost(v3))

In [ ]:
for (verdict, task, what), t in zip(rows, v3):
    if verdict == "REGRESSED":
        print(f"{task}  lost: {what}")
        print(f"  tools: {t['tools']}")
        print(f"  answer: {t['answer']}")
        print()

## สรุป

| | ที่เราเห็นใน lab |
|---|---|
| prompt คือ code | ประกอบจากไฟล์ส่วนย่อย · มี `versions.json` · id = เวอร์ชัน + hash · ดูความต่างแบบ diff |
| trace | หนึ่งบรรทัด JSONL ต่องาน: prompt id, tool, paper ที่อ่าน, paper ที่อ้าง, คำตอบ, token |
| v1 → v2 | เพิ่มประโยคเดียว: อ้างอิงจาก 0/5 เป็น 5/5 · tool และ paper ที่อ่านเหมือนเดิม · ค่าใช้จ่ายใกล้เดิม |
| v2 → v3 | เปลี่ยนแค่ส่วน format: token ขาออกลดราวหนึ่งในสาม แต่ regressed 2 ใน 5 งาน (`[P1, P4]` · ไม่อ่าน paper) |
| อ่าน trace เอง | regression หนึ่งตัวคือพฤติกรรมเปลี่ยนจริง อีกตัวต้องตัดสินว่าจะแก้ prompt หรือแก้ตัวตรวจ |

**แบบฝึกหัด**
1. แก้ regular expression ใน `run_task` ให้นับ `[P1, P4]` ด้วย แล้วรัน `compare(load("v2"), v3)` อีกครั้ง — เหลือ regression กี่ตัว
2. เขียน v4 ที่ยังตอบสั้น แต่บอก model ให้อ่าน paper ก่อนตอบเสมอ แล้วเล่นซ้ำเทียบกับ v2
3. บันทึก v2 สองรอบ — trace ทั้งสองรอบเหมือนกันทุกตัวอักษรไหม ถ้าไม่ ข้อตรวจของเราควรตรวจอะไร และไม่ควรตรวจอะไร

**ถัดไป — Lab 4:** สร้าง harness ตัวเดิมด้วย LangChain แล้วเทียบกับของเราทีละชิ้น